# **Standardizzazione e Confronto Modelli**
Questo notebook esegue un processo uniforme per il caricamento, la pulizia, l'elaborazione e la valutazione dei modelli di machine learning su un dataset fornito.

In [ ]:
# ## 1. Import delle librerie
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Modelli di regressione
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb

# Metriche
from sklearn.metrics import mean_squared_error


In [ ]:
# ## 2. Caricamento del dataset
df = pd.read_excel('aggregati_1.xlsx')

# Mostriamo le prime righe del dataset
display(df.head())


In [ ]:
# ## 3. Rinominare le colonne per maggiore chiarezza
translation_dict = {
    'Unnamed: 0': 'Index',
    'ANNO': 'Year',
    'NUMERO': 'Delivery_Number',
    'VOLUME': 'Volume',
    'PESO': 'Weight',
    'COMMITTENTE': 'Client',
    'LINEA': 'Delivery_Line',
    'QUANTITA_IMBALLI': 'Package_Quantity',
    'VIAGGIO_RITIRO_NUMERO': 'Pickup_Trip_Number',
    'CODICE_DESTINATARIO': 'Recipient_Code',
    'RAGIONE_SOCIALE_destinatario': 'Recipient_Company_Name',
    'INDIRIZZO_destinatario': 'Recipient_Address',
    'CAP_destinatario': 'Recipient_Zip_Code',
    'LOCALITA_destinatario': 'Recipient_City',
    'PROVINCIA_destinatario': 'Recipient_Province',
    'NAZIONE_destinatario': 'Recipient_Country',
    'lat_destinatario': 'Recipient_Latitude',
    'lon_destinatario': 'Recipient_Longitude',
    'CODICE MITTENTE': 'Sender_Code',
    'Ragione_Sociale_MITTENTE': 'Sender_Company_Name',
    'INDIRIZZO_mittente': 'Sender_Address',
    'CAP_mittente': 'Sender_Zip_Code',
    'LOCALITA_mittente': 'Sender_City',
    'PROVINCIA_mittente': 'Sender_Province',
    'NAZIONE_mittente': 'Sender_Country',
    'lat_mittente': 'Sender_Latitude',
    'lon_mittente': 'Sender_Longitude',
    'QUANTITA': 'Quantity'
}

df.rename(columns=translation_dict, inplace=True)
display(df.head())


In [ ]:
# ## 4. Rimozione di colonne non necessarie
columns_to_drop = [
    'Index', 'Quantity', 'Delivery_Number', 'Year', 'Pickup_Trip_Number', 'Client',
    'Recipient_Company_Name', 'Recipient_Address', 'Recipient_Zip_Code', 'Recipient_City', 
    'Recipient_Province', 'Recipient_Country', 'Recipient_Latitude', 'Recipient_Longitude',
    'Sender_Company_Name', 'Sender_Address', 'Sender_Zip_Code', 'Sender_City', 
    'Sender_Province', 'Sender_Country', 'Sender_Latitude', 'Sender_Longitude'
]
df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

print(f"Numero di righe dopo il drop delle colonne inutili: {df.shape[0]}")


In [ ]:
# ## 5. Gestione dei valori mancanti e/o pari a zero per Weight e Volume
missing_volume_or_weight = df['Volume'].isna().sum() + df['Weight'].isna().sum() + (df['Volume'] == 0).sum() + (df['Weight'] == 0).sum()
print(f"Linee con volume o peso mancanti/zero: {missing_volume_or_weight}")

# Rimozione delle righe mancanti/zero
df.dropna(subset=['Volume', 'Weight'], inplace=True)
df = df[(df['Volume'] != 0) & (df['Weight'] != 0)]
print(f"Numero di righe dopo rimozione di NaN o zeri: {df.shape[0]}")


In [ ]:
# ## 6. Rimozione degli outlier usando IQR per le sole colonne numeriche
numeric_cols = ['Volume', 'Weight']
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1

df = df[~((df[numeric_cols] < (Q1 - 1.5 * IQR)) | (df[numeric_cols] > (Q3 + 1.5 * IQR))).any(axis=1)]
print(f"Numero di righe dopo rimozione outlier (IQR): {df.shape[0]}")


In [ ]:
# ## 7. Encoding delle variabili categoriche
label_encoder_recipient = LabelEncoder()
label_encoder_sender = LabelEncoder()

df['Recipient_Code'] = label_encoder_recipient.fit_transform(df['Recipient_Code'])
df['Sender_Code'] = label_encoder_sender.fit_transform(df['Sender_Code'])
print("Encoding completato per Recipient_Code e Sender_Code")


In [ ]:
# ## 8. Esplorazione iniziale e statistiche descrittive
print(df.describe(include='all'))

# ## 9. Suddivisione del dataset
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42)
eval_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
print(f"Train set: {train_df.shape[0]} righe")
